In [1]:
def get_image_path(row, img_lookup):
    return img_lookup.get(row["filename"])

In [2]:
def get_channel_indices(row, n_channels=4):
    channel_map = {}

    for i in range(n_channels):
        name = row.get(f"ch{i}")
        if name:
            channel_map[name] = i
    return channel_map

In [3]:
import numpy as np
from cellpose import models

def run_cellpose_two_channel(
    img,
    nuc_ch,
    cyto_ch,
    *,
    model=None,
    gpu=True,
    batch_size=32,
    diameter=120,
    flow_threshold=0.4,
    cellprob_threshold=0.0,
    tile_norm_blocksize=0,
):
    """
    Run Cellpose on a 2-channel stack (e.g., nucleus + cytoplasm) for Cellpose v4+,
    where the old `channels=` argument is deprecated.

    Parameters
    ----------
    img : np.ndarray
        Image array. Expected shape (C, Y, X) or (Y, X).
    nuc_ch : int | None
        Index for nucleus channel if img is (C, Y, X). Ignored if img is 2D.
    cyto_ch : int | None
        Index for cytoplasm/cell-body channel if img is (C, Y, X). Ignored if img is 2D.
    model : cellpose.models.CellposeModel | None
        If None, creates a new CellposeModel(gpu=gpu).
    gpu : bool
        Whether to use GPU when creating a model.
    batch_size : int
        Batch size for eval.
    diameter : float | int | None
        Expected object diameter (pixels). None lets Cellpose estimate.
    flow_threshold : float
        Flow threshold.
    cellprob_threshold : float
        Cell probability threshold.
    tile_norm_blocksize : int
        Passed to normalize={"tile_norm_blocksize": ...}.
    return_stack : bool
        If True, also return the constructed 2-channel stack.

    Returns
    -------
    masks : np.ndarray
    flows : list/tuple
    styles : np.ndarray
    diams : float or np.ndarray
    (optional) stack : np.ndarray
        The 2-channel stack used as input (2, Y, X).
    """
    if model is None:
        model = models.CellposeModel(gpu=gpu)

    if img.ndim == 3:
        nuc = img[nuc_ch]
        cyto = img[cyto_ch]
    elif img.ndim == 2:
        # If user passes a 2D image, treat it as "cyto" and set nuc equal to cyto,
        # or raise—here we raise because the function is intended for 2-channel use.
        raise ValueError("img is 2D; this function expects a channel-first (C, Y, X) array.")
    else:
        raise ValueError(f"Unsupported img.ndim={img.ndim}; expected 2 or 3.")

    stack = np.stack([nuc, cyto], axis=0)

    masks, flows, styles = model.eval(
        stack,
        batch_size=batch_size,
        diameter=diameter,
        flow_threshold=flow_threshold,
        cellprob_threshold=cellprob_threshold,
        normalize={"tile_norm_blocksize": tile_norm_blocksize},
    )

    return masks, flows, styles


In [4]:
import numpy as np
from scipy.stats import norm
from scipy.ndimage import gaussian_laplace
from scipy import ndimage as ndi
from skimage import filters, morphology

def make_lysosome_mask(
    channel,
    intensity_scaling_param=(3, 19),
    min_lysosome_area=5,
    blur_sigma=1,
    log_params=((2.5, 0.07), (2.0, 0.04), (1.0, 0.01)),
    vesselness_sigma=(1,),
    vesselness_cutoff=0.15
):
    """
    Create a lysosome mask using multi-scale LoG blob detection + Frangi vesselness,
    followed by hole-filling and small-object removal.

    Parameters
    ----------
    channel : np.ndarray
        Channel index to use if img is channel-first (C, Y, X). If img is 2D, use None.
    intensity_scaling_param : (float, float)
        (low_sigma_mult, high_sigma_mult) for normal-fit contrast stretching.
    min_lysosome_area : int
        Minimum object size (pixels) to keep.
    blur_sigma : float
        Gaussian blur sigma applied after normalization.
    log_params : iterable of (sigma, cutoff)
        LoG scales and thresholds.
    vesselness_sigma : iterable of float
        Sigmas passed to skimage.filters.frangi.
    vesselness_cutoff : float
        Threshold applied to Frangi response.

    Returns
    -------
    lysosome_mask : np.ndarray (bool)
        Final binary mask.
    """

    # normalize image
    m, s = norm.fit(channel.ravel())
    low_mult, high_mult = intensity_scaling_param

    stretch_min = max(m - low_mult * s, float(np.min(channel)))
    stretch_max = min(m + high_mult * s, float(np.max(channel)))

    # guard against divide-by-zero if image is constant-ish
    denom = (stretch_max - stretch_min)
    if denom <= 0:
        image_norm = np.zeros_like(channel, dtype=float)
    else:
        channel_n = np.clip(channel, stretch_min, stretch_max)
        image_norm = (channel_n - stretch_min) / denom

    # blur normalized image
    blurred = filters.gaussian(image_norm, sigma=blur_sigma)

    # LoG blobs at multiple scales
    log_masks = []
    log_responses = []
    for sigma, cutoff in log_params:
        log = -1.0 * (sigma**2) * gaussian_laplace(blurred, sigma=sigma)
        log_responses.append(log)
        log_masks.append(log > cutoff)

    log_mask = np.logical_or.reduce(log_masks)

    # vesselness (Frangi)
    vesselness_response = filters.frangi(
        blurred, sigmas=tuple(vesselness_sigma), black_ridges=False
    )
    vesselness_mask = vesselness_response > vesselness_cutoff

    # combine + postprocess
    combined = np.logical_or(log_mask, vesselness_mask)
    combined = ndi.binary_fill_holes(combined)
    lysosome_mask = morphology.remove_small_objects(
        combined.astype(bool), min_size=min_lysosome_area
    )
    
    return lysosome_mask

In [5]:
from scipy.ndimage import gaussian_laplace
from skimage import filters, morphology
from scipy import ndimage as ndi
import numpy as np
from scipy.stats import norm

def make_prab10_mask(
    channel,
    intensity_scaling_param=(3, 19),
    min_prab10_area=5,
    blur_sigma=1,
    log_params=((3.0, 0.07), (2.0, 0.04), (1.0, 0.01)),
    vesselness_sigma=(1,),
    vesselness_cutoff=0.15
):
    """
    Create a mask for phosphorylated RAB10 using multi-scale LoG blob detection + Frangi vesselness,
    followed by hole-filling and small-object removal.

    Parameters
    ----------
    channel : np.ndarray
        Channel index to use if img is channel-first (C, Y, X). If img is 2D, use None.
    intensity_scaling_param : (float, float)
        (low_sigma_mult, high_sigma_mult) for normal-fit contrast stretching.
    min_prab10_area : int
        Minimum object size (pixels) to keep.
    blur_sigma : float
        Gaussian blur sigma applied after normalization.
    log_params : iterable of (sigma, cutoff)
        LoG scales and thresholds.
    vesselness_sigma : iterable of float
        Sigmas passed to skimage.filters.frangi.
    vesselness_cutoff : float
        Threshold applied to Frangi response.

    Returns
    -------
    prab10_mask : np.ndarray (bool)
        Final binary mask.
    """

    # normalize image
    m, s = norm.fit(channel.ravel())
    low_mult, high_mult = intensity_scaling_param

    stretch_min = max(m - low_mult * s, float(np.min(channel)))
    stretch_max = min(m + high_mult * s, float(np.max(channel)))

    # guard against divide-by-zero if image is constant-ish
    denom = (stretch_max - stretch_min)
    if denom <= 0:
        image_norm = np.zeros_like(channel, dtype=float)
    else:
        channel_n = np.clip(channel, stretch_min, stretch_max)
        image_norm = (channel_n - stretch_min) / denom

    # blur normalized image
    blurred = filters.gaussian(image_norm, sigma=blur_sigma)

    # LoG blobs at multiple scales
    log_masks = []
    log_responses = []
    for sigma, cutoff in log_params:
        log = -1.0 * (sigma**2) * gaussian_laplace(blurred, sigma=sigma)
        log_responses.append(log)
        log_masks.append(log > cutoff)

    log_mask = np.logical_or.reduce(log_masks)

    # vesselness (Frangi)
    vesselness_response = filters.frangi(
        blurred, sigmas=tuple(vesselness_sigma), black_ridges=False
    )
    vesselness_mask = vesselness_response > vesselness_cutoff

    # combine + postprocess
    combined = np.logical_or(log_mask, vesselness_mask)
    combined = ndi.binary_fill_holes(combined)
    prab10_mask = morphology.remove_small_objects(
        combined.astype(bool), min_size=min_prab10_area

    )
    
    return prab10_mask

In [6]:
import numpy as np
from scipy import ndimage as ndi

def label_objects_within_cells(
    cell_masks,
    obj_mask,
    *,
    connectivity=1,
    start_label=1,
):
    """
    For each cell ID in cell_masks, label connected components of obj_mask within that cell,
    write them into a global label image with unique labels, and return a mapping of
    object_label -> cell_id.

    Parameters
    ----------
    cell_masks : np.ndarray (int)
        Labeled cell segmentation (0=background, 1..N=cells)
    obj_mask : np.ndarray (bool or int)
        Binary mask of objects (lysosomes, pRAB10 puncta, etc.)
    connectivity : int
        Connectivity for ndi.label: 1 = 4-connectivity (2D), 2 = 8-connectivity (2D)
    start_label : int
        Starting label for global objects.

    Returns
    -------
    labels_global : np.ndarray (int32)
        Global labeled image of all objects with unique IDs
    parent_cell : dict[int, int]
        Mapping from global object label -> parent cell id
    next_label : int
        Next available label (useful if chaining)
    """
    obj_mask = obj_mask.astype(bool, copy=False)
    labels_global = np.zeros_like(obj_mask, dtype=np.int32)
    parent_cell = {}

    cell_ids = np.unique(cell_masks)
    cell_ids = cell_ids[cell_ids != 0]

    current_label = int(start_label)

    # structure defines connectivity
    structure = ndi.generate_binary_structure(rank=2, connectivity=connectivity)

    for cid in cell_ids:
        single_cell = (cell_masks == cid)
        obj_in_cell = obj_mask & single_cell

        labeled_in_cell, n = ndi.label(obj_in_cell, structure=structure)
        if n == 0:
            continue

        # iterate over component labels 1..n
        for local_label in range(1, n + 1):
            labels_global[labeled_in_cell == local_label] = current_label
            parent_cell[current_label] = int(cid)
            current_label += 1

    return labels_global, parent_cell, current_label


In [7]:
import pandas as pd
from skimage.measure import regionprops_table

def object_props_dataframe(
    labels_global,
    intensity_image,
    parent_cell_map,
    *,
    image_id=None,
    properties=("label", "area", "min_intensity", "mean_intensity", "max_intensity"),
):
    """
    Build a DataFrame of regionprops for labeled objects, and add cell_id + optional image_id.
    """
    props = regionprops_table(
        labels_global,
        intensity_image=intensity_image,
        properties=properties,
    )
    df = pd.DataFrame(props)
    df["cell_id"] = df["label"].map(parent_cell_map)

    if image_id is not None:
        df.insert(0, "image_id", image_id)

    return df


In [8]:
import os

def quantify_objects_per_cell(
    *,
    img,
    img_path,
    cell_masks,
    lysosome_mask,
    prab10_mask,
    lyso_ch,
    prab10_ch,
    connectivity=1,
    properties=("label", "area", "min_intensity", "mean_intensity", "max_intensity"),
    return_label_images=False,
):
    """
    Labels lysosomes and pRAB10 objects within each cell, then measures intensity/area props.

    Returns
    -------
    lys_df, prab_df
    (optional) lys_labels_global, prab_labels_global, lys_parent_cell, prab_parent_cell
    """
    image_id = os.path.basename(img_path)

    # label objects within cells
    lys_labels_global, lys_parent_cell, _ = label_objects_within_cells(
        cell_masks, lysosome_mask, connectivity=connectivity, start_label=1
    )
    prab_labels_global, prab_parent_cell, _ = label_objects_within_cells(
        cell_masks, prab10_mask, connectivity=connectivity, start_label=1
    )

    # intensity images
    lys_int = img[lyso_ch]
    prab_int = img[prab10_ch]

    # props -> dfs
    lys_df = object_props_dataframe(
        lys_labels_global,
        lys_int,
        lys_parent_cell,
        image_id=image_id,
        properties=properties,
    )
    prab_df = object_props_dataframe(
        prab_labels_global,
        prab_int,
        prab_parent_cell,
        image_id=image_id,
        properties=properties,
    )

    if return_label_images:
        return (
            lys_df,
            prab_df,
            lys_labels_global,
            prab_labels_global,
            lys_parent_cell,
            prab_parent_cell,
        )

    return lys_df, prab_df


In [9]:
import os
import numpy as np
import pandas as pd
from skimage.filters import threshold_otsu

def per_cell_colocalization(
    cell_masks,
    lys_img,
    prab_img,
    *,
    image_id=None,
    img_path=None,
    min_pixels=50,
    threshold_method="otsu",
):
    """
    Compute per-cell colocalization metrics between two channels:
      - Pearson correlation
      - Manders M1 (lys in prab) and M2 (prab in lys)
    Thresholds are computed per-cell (default: Otsu) on pixel intensities inside each cell.

    Parameters
    ----------
    cell_masks : np.ndarray (int)
        Labeled cell segmentation (0=background).
    lys_img : np.ndarray
        Lysosome intensity image (Y, X).
    prab_img : np.ndarray
        pRAB10 intensity image (Y, X).
    image_id : str | None
        Optional image identifier to insert into output.
    img_path : str | None
        If image_id is None and img_path is provided, uses basename(img_path).
    min_pixels : int
        Skip cells with fewer pixels than this.
    threshold_method : {"otsu"}
        Currently supports "otsu".

    Returns
    -------
    pd.DataFrame
        One row per cell with colocalization metrics.
    """
    if image_id is None and img_path is not None:
        image_id = os.path.basename(img_path)

    lys = np.asarray(lys_img, dtype=np.float64)
    prab = np.asarray(prab_img, dtype=np.float64)

    if lys.shape != prab.shape or lys.shape != cell_masks.shape:
        raise ValueError(
            f"Shape mismatch: lys {lys.shape}, prab {prab.shape}, cell_masks {cell_masks.shape}"
        )

    cell_ids = np.unique(cell_masks)
    cell_ids = cell_ids[cell_ids != 0]

    rows = []

    for cid in cell_ids:
        m = (cell_masks == cid)
        npx = int(m.sum())
        if npx < min_pixels:
            continue

        a = lys[m]
        b = prab[m]

        # Pearson r (guard constant arrays)
        if a.std() == 0 or b.std() == 0:
            pearson_r = np.nan
        else:
            pearson_r = float(np.corrcoef(a, b)[0, 1])

        # Per-cell thresholds
        if threshold_method == "otsu":
            # Otsu can fail if array has 1 unique value; fall back to 0.0
            tA = float(threshold_otsu(a)) if np.unique(a).size > 1 else 0.0
            tB = float(threshold_otsu(b)) if np.unique(b).size > 1 else 0.0
        else:
            raise ValueError(f"Unsupported threshold_method: {threshold_method}")

        a_pos = a > tA
        b_pos = b > tB

        denomA = a[a_pos].sum()
        denomB = b[b_pos].sum()

        # Manders coefficients
        M1 = float(a[a_pos & b_pos].sum() / denomA) if denomA > 0 else np.nan  # lys in prab
        M2 = float(b[b_pos & a_pos].sum() / denomB) if denomB > 0 else np.nan  # prab in lys

        rows.append({
            "cell_id": int(cid),
            "cell_pixels": npx,
            "pearson_r": pearson_r,
            "manders_M1_lys_in_prab": M1,
            "manders_M2_prab_in_lys": M2,
            "threshold_cutoff_lys": tA,
            "threshold_cutoff_prab": tB,
        })

    df = pd.DataFrame(rows)
    if image_id is not None:
        df.insert(0, "image_id", image_id)

    return df

In [10]:
import tifffile
import traceback

def process_single_image(row, img_lookup):
    """
    Process one image defined by a samplesheet row.
    Returns (lys_df, prab_df, cell_coloc_df) or (None, None, None) on failure.
    """
    try:
        img_path = img_lookup.get(row["filename"])
        if img_path is None:
            raise FileNotFoundError(f"Image not found: {row['filename']}")

        # channel indices
        ch_index = get_channel_indices(row)
        dapi_ch = ch_index["DAPI"]
        lyso_ch = ch_index["LAMP1"]
        prab10_ch = ch_index["pRAB10"]
        gfap_ch = ch_index["GFAP"]

        # load image
        img = tifffile.imread(img_path)

        # segment cells
        cell_masks, flows, styles, diams = run_cellpose_two_channel(
            img,
            nuc_ch=dapi_ch,
            cyto_ch=gfap_ch,
        )

        # segment lysosomes and prab10
        lysosome_mask = make_lysosome_mask(img[lyso_ch])
        prab10_mask = make_prab10_mask(img[prab10_ch])

        # extract per cell metrics
        lys_df, prab_df = quantify_objects_per_cell(
            img=img,
            img_path=img_path,
            cell_masks=cell_masks,
            lysosome_mask=lysosome_mask,
            prab10_mask=prab10_mask,
            lyso_ch=lyso_ch,
            prab10_ch=prab10_ch,
            connectivity=1,
        )

        # extract per cell colocalization
        cell_coloc_df = per_cell_colocalization(
            cell_masks=cell_masks,
            lys_img=img[lyso_ch],
            prab_img=img[prab10_ch],
            img_path=img_path,
            min_pixels=50,
        )

        return lys_df, prab_df, cell_coloc_df

    except Exception as e:
        print(f"❌ Failed on {row['filename']}: {e}")
        traceback.print_exc()
        return None, None, None


In [11]:
import tifffile 
import pandas as pd
from pathlib import Path

samplesheet = pd.read_csv("/Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Hannah_RepresentativeImages/hb_samplesheet.csv")
img_dir = Path("/Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Hannah_RepresentativeImages")

imgs = list(img_dir.glob("*.tif"))

img_lookup = {p.name: p for p in imgs}
# for img in imgs?

test = samplesheet.iloc[4] #img index
fname = test["filename"]

img_path = img_lookup.get(fname)

samplesheet["image_path"] = samplesheet.apply(
    get_image_path, axis=1, img_lookup=img_lookup
)

ch_index = get_channel_indices(test)

dapi_ch = ch_index["DAPI"]
gfap_ch = ch_index["GFAP"]
lyso_ch = ch_index["LAMP1"]
prab10_ch = ch_index["pRAB10"]

img = tifffile.imread(img_path)

cell_masks, flows, styles = run_cellpose_two_channel(
    img,
    nuc_ch=dapi_ch,
    cyto_ch=prab10_ch,
)

lysosome_mask = make_lysosome_mask(img[lyso_ch])
prab10_mask = make_prab10_mask(img[prab10_ch])

lys_df, prab_df = quantify_objects_per_cell(
    img=img,
    img_path=img_path,
    cell_masks=cell_masks,
    lysosome_mask=lysosome_mask,
    prab10_mask=prab10_mask,
    lyso_ch=lyso_ch,
    prab10_ch=prab10_ch,
    connectivity=1,
)

cell_coloc_df = per_cell_colocalization(
    cell_masks=cell_masks,
    lys_img=img[lyso_ch],
    prab_img=img[prab10_ch],
    img_path=img_path,
    min_pixels=50,
)


In [12]:
all_lys = []
all_prab = []
all_cell_coloc = []

for _, row in samplesheet.iterrows():
    lys_df, prab_df, cell_coloc_df = process_single_image(row, img_lookup)

    if lys_df is None:
        continue  # skip failed images

    all_lys.append(lys_df)
    all_prab.append(prab_df)
    all_cell_coloc.append(cell_coloc_df)

lys_all_df = pd.concat(all_lys, ignore_index=True)
prab_all_df = pd.concat(all_prab, ignore_index=True)
cell_coloc_all_df = pd.concat(all_cell_coloc, ignore_index=True)

print(lys_all_df.shape, prab_all_df.shape, cell_coloc_all_df.shape)

❌ Failed on 20240729_WT_ACs_pR10_647_Lamp1_568_GFAP488_NG_002_MIP.tif: not enough values to unpack (expected 4, got 3)


Traceback (most recent call last):
  File "/var/folders/xx/bl1zrt2j5z18wq1lwt297lb9qnfc7z/T/ipykernel_40886/1243341571.py", line 25, in process_single_image
    cell_masks, flows, styles, diams = run_cellpose_two_channel(
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: not enough values to unpack (expected 4, got 3)


❌ Failed on 20240729_WT_ACs_pR10_647_Lamp1_568_GFAP488_Unt010_MIP.tif: not enough values to unpack (expected 4, got 3)


Traceback (most recent call last):
  File "/var/folders/xx/bl1zrt2j5z18wq1lwt297lb9qnfc7z/T/ipykernel_40886/1243341571.py", line 25, in process_single_image
    cell_masks, flows, styles, diams = run_cellpose_two_channel(
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: not enough values to unpack (expected 4, got 3)


KeyboardInterrupt: 

In [12]:
import napari

viewer = napari.Viewer()

viewer.add_image(img[dapi_ch], 
                 name='DAPI',
                 blending="additive",
                 colormap="blue"
                 )

viewer.add_image(img[gfap_ch], 
                 name='GFAP',
                 blending="additive",
                 colormap="green"
                 )

viewer.add_image(img[lyso_ch], 
                 name='LAMP1',
                 blending="additive",
                 colormap="red")

viewer.add_image(img[prab10_ch], 
                 name='pRAB10',
                 blending="additive",
                 colormap="magenta")

viewer.add_labels(cell_masks,
                  name = "Cell mask")

viewer.add_labels(lysosome_mask, 
                 name='LYSOSOME MASK')

viewer.add_labels(prab10_mask, 
                 name='PRAB10 MASK')

<Labels layer 'PRAB10 MASK' at 0x4264b02d0>